In [4]:
import json
import random
import uuid
import os
from faker import Faker
from datetime import datetime, timedelta

fake = Faker()
OUTPUT_DIR = "C:/Users/julie/Data Eng/Raw/"

EVENTS = ["order_created", "driver_assigned", "pickup", "delivery_success", "delivery_failed"]
DRIVER_IDS = [f"d{str(i).zfill(3)}" for i in range(1, 11)]
CLIENT_IDS = [f"c{str(i).zfill(3)}" for i in range(1, 21)]


def generate_event(delivery_id, step, base_time):
    client_id = random.choice(CLIENT_IDS)
    driver_id = random.choice(DRIVER_IDS)
    timestamp = base_time + timedelta(minutes=step * random.randint(1, 10))
    location = fake.local_latlng(country_code="FR", coords_only=True)

    event = {
        "event": EVENTS[step],
        "delivery_id": delivery_id,
        "timestamp": timestamp.isoformat(),
        "client_id": client_id,
        "driver_id": driver_id,
        "location": {
            "lat": float(location[0]),
            "lon": float(location[1])
        }
    }

    # Ajouter durée uniquement pour les livraisons réussies/échouées
    if EVENTS[step] in ["delivery_success", "delivery_failed"]:
        event["duration_minutes"] = random.randint(10, 60)
        if EVENTS[step] == "delivery_failed":
            event["failure_reason"] = random.choice(["client_absent", "adresse_invalide", "annulation"])

    return event


def simulate_deliveries(n=100):
    for _ in range(n):
        delivery_id = str(uuid.uuid4())
        base_time = datetime.now() - timedelta(days=random.randint(0, 30))

        steps = random.choices(range(3, 5), weights=[0.2, 0.8])[0]  # 80% succès, 20% échec
        for step in range(steps):
            event = generate_event(delivery_id, step, base_time)

            filename = f"{OUTPUT_DIR}event_{delivery_id}_{event['event']}.json"
            with open(filename, "w") as f:
                json.dump(event, f, indent=2)


if __name__ == "__main__":
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    simulate_deliveries(100)
    print("✅ 100 livraisons simulées générées")


✅ 100 livraisons simulées générées
